In [1]:
import pandas as pd
import sqlite3

In [2]:
sqliteConnection = sqlite3.connect('olist.db')

### Building my first Common Table Expression (CTE)

first step is to join the customers table with orders and the order payments 


In [14]:
query = """
WITH raw_metrics(customer_unique_id,frequency,recency,monetary) AS (
    SELECT  c.customer_unique_id,COUNT(DISTINCT o.order_id),MAX(o.order_purchase_timestamp),SUM(p.payment_value)
    FROM orders o
    JOIN order_payments p
    ON o.order_id = p.order_id
    JOIN customers c
    ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
    )
SELECT * FROM raw_metrics
LIMIT 5;
"""
pd.read_sql_query(query,sqliteConnection)

,customer_unique_id,frequency,recency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,27.19
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,196.89


1 - We have the date of their last purchase, but "Recency" is usually expressed in days. We need to compare their last purchase date to the very last day of data in the entire database (the "anchor date")
2 - Assign scores from 1 to 5 to the customers based on their metrics (we will have 3 scores for each metric f_score, r_score,m_score)
3 - Assign a label to each customer based on their scores for each category
4 - Aggregate the number of customers for each category 

In [27]:
query = """
WITH raw_metrics(customer_unique_id,frequency,recency,monetary) AS (
    SELECT  c.customer_unique_id,COUNT(DISTINCT o.order_id),MAX(o.order_purchase_timestamp),SUM(p.payment_value)
    FROM orders o
    JOIN order_payments p
    ON o.order_id = p.order_id
    JOIN customers c
    ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
    ),
anchor_date(last_date) AS (
    SELECT MAX(o.order_purchase_timestamp)
    FROM orders o
    
),
customer_metrics (customer_unique_id,frequency,monetary,recency_days) AS (  
SELECT r.customer_unique_id,r.frequency,r.monetary, CAST(julianday(d.last_date) - julianday(r.recency) AS INTEGER) AS recency_days
FROM raw_metrics r, anchor_date d
),

customer_scores (customer_unique_id,frequency,monetary,recency_days,f_score,m_score,r_score) AS (
    SELECT *,
    NTILE(5) OVER (ORDER BY frequency ASC) f_score,
    NTILE(5) OVER (ORDER BY monetary ASC) m_score,
    NTILE(5) OVER (ORDER BY recency_days DESC) r_score
    FROM customer_metrics
),
customers_labels(unique_id,label) AS (
SELECT customer_unique_id, 
CASE
    WHEN s.f_score>=4 AND s.m_score>=4 AND s.r_score>=4 THEN 'Champion'
    WHEN s.f_score>=4 AND s.m_score>=3 AND s.r_score>=3 THEN 'Loyal'
    WHEN s.f_score<=2  AND s.r_score>=4 THEN 'Recent customer'
    WHEN s.f_score>=3 AND s.m_score>=3 AND s.r_score<=2 THEN 'At risk'
    WHEN s.f_score<=2 AND s.m_score<=2 AND s.r_score<=2 THEN 'Lost'
    ELSE 'Other'
    END as customer_label
FROM customer_scores s
)

SELECT label, COUNT(label) as total_customers
FROM customers_labels
GROUP BY label
"""
pd.read_sql_query(query,sqliteConnection)

,label,total_customers
0,At risk,22467
1,Champion,15554
2,Lost,15784
3,Loyal,7792
4,Other,19702
5,Recent customer,14796


In [28]:
sqliteConnection.close()